<center><img src="MKn_Staffelter_Hof.jpeg" alt="Picture of old business"</center>
<!--Image Credit: Martin Kraft https://commons.wikimedia.org/wiki/File:MKn_Staffelter_Hof.jpg -->

Staffelter Hof Winery is Germany's oldest business, established in 862 under the Carolingian dynasty. It has continued to serve customers through dramatic changes in Europe, such as the Holy Roman Empire, the Ottoman Empire, and both world wars. What characteristics enable a business to stand the test of time?

To help answer this question, BusinessFinancing.co.uk researched the oldest company still in business in **almost** every country and compiled the results into several CSV files. This dataset has been cleaned.

Having useful information in different files is a common problem. While it's better to keep different types of data separate for data storage, you'll want all the data in one place for analysis. You'll use joining and data manipulation to work with this data and better understand the world's oldest businesses.

## The Data
`businesses` and `new_businesses`
|Column|Description|
|------|-----------|
|`business`|Name of the business (varchar)|
|`year_founded`|Year the business was founded (int)|
|`category_code`|Code for the business category (varchar)|
|`country_code`|ISO 3166-1 three-letter country code (char)|
---
`countries`
|Column|Description|
|------|-----------|
|`country_code`|ISO 3166-1 three-letter country code (varchar)|
|`country`|Name of the country (varchar)|
|`continent`|Name of the continent the country exists in (varchar)|
---
`categories`
|Column|Description|
|------|-----------|
|`category_code`|Code for the business category (varchar)|
|`category`|Description of the business category (varchar)|

In [15]:
SELECT *
FROM businesses
LIMIT 5;

,business,year_founded,category_code,country_code
0,Hamoud Boualem,1878,CAT11,DZA
1,Communauté Électrique du Bénin,1968,CAT10,BEN
2,Botswana Meat Commission,1965,CAT1,BWA
3,Air Burkina,1967,CAT2,BFA
4,Brarudi,1955,CAT9,BDI


In [16]:
SELECT *
FROM countries
LIMIT 5;

,country_code,country,continent
0,AFG,Afghanistan,Asia
1,AGO,Angola,Africa
2,ALB,Albania,Europe
3,AND,Andorra,Europe
4,ARE,United Arab Emirates,Asia


In [17]:
SELECT *
FROM categories
LIMIT 5;

,category_code,category
0,CAT1,Agriculture
1,CAT2,Aviation & Transport
2,CAT3,Banking & Finance
3,CAT4,"Cafés, Restaurants & Bars"
4,CAT5,Conglomerate


In [18]:
(
SELECT 
	MIN(b.year_founded)
FROM businesses AS b
INNER JOIN countries AS c
USING (country_code)
GROUP BY c.continent
)

,min
0,1772
1,578
2,1565
3,1534
4,803
5,1809


In [19]:
-- What is the oldest business on each continent?
SELECT
	s1.business,
	s1.year_founded,
	s1.country,
	s1.continent
FROM (
	SELECT
		b.country_code,
		b.business,
		b.year_founded,
		c.country,
		c.continent
	FROM businesses AS b
	INNER JOIN countries AS c
	USING(country_code)
) AS s1
INNER JOIN (
	SELECT 
	c.continent,	
	MIN(b.year_founded) AS oldest_year
	FROM businesses AS b
	INNER JOIN countries AS c
	USING (country_code)
	GROUP BY c.continent
) AS s2
ON s1.continent = s2.continent
	AND s1.year_founded = s2.oldest_year
ORDER BY s1.continent ASC;

,business,year_founded,country,continent
0,Mauritius Post,1772,Mauritius,Africa
1,Kongō Gumi,578,Japan,Asia
2,St. Peter Stifts Kulinarium,803,Austria,Europe
3,La Casa de Moneda de México,1534,Mexico,North America
4,Australia Post,1809,Australia,Oceania
5,Casa Nacional de Moneda,1565,Peru,South America


In [20]:
-- How many countries per continent lack data on the oldest businesses
-- Does including the `new_businesses` data change this?
SELECT 
	c.continent,
	COUNT(c.country) AS countries_without_businesses
FROM countries AS c
LEFT JOIN (
		SELECT *
		FROM businesses
		UNION ALL
		SELECT *
		FROM new_businesses
		WHERE business IS NULL) AS b
USING(country_code)
WHERE b.business IS NULL
GROUP BY c.continent;

,continent,countries_without_businesses
0,Africa,3
1,Asia,7
2,Europe,2
3,North America,6
4,Oceania,11
5,South America,3


In [21]:
-- Which business categories are best suited to last over the course of centuries?
SELECT 
	c1.continent,
	c2.category,
	MIN(b.year_founded) AS year_founded
FROM businesses AS b
INNER JOIN countries AS c1
	ON b.country_code = c1.country_code
INNER JOIN categories AS c2
	ON b.category_code = c2.category_code 
GROUP BY 
	c1.continent,
	c2.category
ORDER BY year_founded ASC;


,continent,category,year_founded
0,Asia,Construction,578
1,Europe,"Cafés, Restaurants & Bars",803
2,Europe,"Distillers, Vintners, & Breweries",862
3,Europe,Manufacturing & Production,864
4,Asia,"Cafés, Restaurants & Bars",1153
5,Europe,Agriculture,1218
6,Europe,Tourism & Hotels,1230
7,Europe,Mining,1248
8,Europe,Medical,1422
9,Europe,Postal Service,1520
